# OUARTI — Versions, transferts et retour a un agent precedent

Aucune connexion au jeu. **REGLAGES**, puis **Tout executer**. Par defaut, seul l'inventaire est affiche.

- `archiver` : conserver une version exacte du ZIP et des metadonnees.
- `transferer_stage` : creer un nouvel agent pour le stage suivant, sans toucher a la source.
- `revenir_version` : copier une ancienne version vers un NOUVEAU nom, en gardant son stage et son compteur.

Copier une valeur `source_selectionnable` de l'inventaire dans `MODELE_SOURCE`. Pour reutiliser l'agent restaure, choisir son nom et son stage dans les reglages du notebook 02 ou 03. Pour l'evaluer, choisir le meme nom et le meme stage dans 04.01.

Les versions evitent les ecrasements par le code du projet, sans remplacer une sauvegarde sur un autre disque : `Code/models/` etant ignore par Git, un push ne sauvegarde pas les agents.


In [ ]:
# REGLAGES — aucune conduite ni apprentissage ici.
ACTIVER_OPERATION = False
OPERATION = 'revenir_version'  # archiver, transferer_stage, revenir_version
MODELE_SOURCE = 'ppo_vision_checkpoint_v3_stage1'  # ou versions/.../agent
NOM_MODELE_CIBLE = 'ppo_vision_checkpoint_v3_stage1_retour_01'
STAGE_CIBLE = 2              # utilise seulement pour transferer_stage


In [ ]:
from pathlib import Path
import json, time, uuid, shutil, os
from IPython.display import display
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'Code'/'models').is_dir()),None)
if REPO is None:raise FileNotFoundError('Ouvrir le depot ou un sous-dossier.')
PROJECT=REPO/'Code'
import hashlib, re


def empreinte_fichier(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()


def prefix_modele(selection):
    path=Path(selection)
    if not path.is_absolute():path=PROJECT/'models'/path
    if path.suffix=='.zip':path=path.with_suffix('')
    return path.resolve()


def fichiers_modele(prefix):
    prefix=Path(prefix)
    return prefix.with_suffix('.zip'),prefix.with_name(prefix.name+'_meta.json')


def lire_modele(selection):
    prefix=prefix_modele(selection);model,metadata=fichiers_modele(prefix)
    if not model.is_file() or not metadata.is_file():
        raise FileNotFoundError('ZIP et metadonnees requis : '+str(prefix))
    versions=(PROJECT/'models'/'versions').resolve()
    if versions in prefix.parents:
        manifest_path=prefix.parent/'manifest.json'
        if not manifest_path.is_file():raise RuntimeError('Archive sans manifeste : '+str(prefix))
        manifest=json.loads(manifest_path.read_text(encoding='utf8'))
        if empreinte_fichier(model)!=manifest.get('zip_sha256') or empreinte_fichier(metadata)!=manifest.get('meta_sha256'):
            raise RuntimeError('Archive alteree, lecture refusee : '+str(prefix))
    return prefix,json.loads(metadata.read_text(encoding='utf8'))


def verifier_destination_modifiable(prefix):
    prefix=Path(prefix).resolve()
    versions=(PROJECT/'models'/'versions').resolve()
    if prefix==versions or versions in prefix.parents:
        raise RuntimeError('Les archives versions sont protegees. Creer un agent de travail sous un nouveau nom.')


def destination_agent(nom):
    if not isinstance(nom,str) or re.fullmatch(r'[A-Za-z0-9_-]+',nom) is None:
        raise ValueError('Nom de modele : lettres, chiffres, tirets et underscores uniquement.')
    prefix=(PROJECT/'models'/nom).resolve()
    verifier_destination_modifiable(prefix)
    return prefix


def archiver_agent(selection,motif='avant_modification'):
    prefix,meta=lire_modele(selection)
    model,metadata=fichiers_modele(prefix)
    hashes={'zip_sha256':empreinte_fichier(model),'meta_sha256':empreinte_fichier(metadata)}
    versions=PROJECT/'models'/'versions'
    # The exact same ZIP/metadata pair only needs one archive.
    for existing in versions.glob('*/*/manifest.json'):
        manifest=json.loads(existing.read_text(encoding='utf8'))
        if all(manifest.get(k)==v for k,v in hashes.items()):
            archived=existing.parent/'agent'
            a,b=fichiers_modele(archived)
            if not a.is_file() or not b.is_file() or empreinte_fichier(a)!=hashes['zip_sha256'] or empreinte_fichier(b)!=hashes['meta_sha256']:
                raise RuntimeError('Archive alteree : '+str(existing.parent))
            return archived
    label=re.sub(r'[^A-Za-z0-9_-]','_',str(meta.get('model',prefix.name)))
    folder=versions/label/(time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8])
    folder.mkdir(parents=True,exist_ok=False)
    archived=folder/'agent';a,b=fichiers_modele(archived)
    shutil.copy2(model,a);shutil.copy2(metadata,b)
    if empreinte_fichier(a)!=hashes['zip_sha256'] or empreinte_fichier(b)!=hashes['meta_sha256']:
        raise RuntimeError('Copie archive non conforme : '+str(folder))
    manifest={'version_format':1,'source':str(prefix),'motif':motif,'created_at':time.time(),
        'model':meta.get('model',prefix.name),'curriculum_stage':meta.get('curriculum_stage'),
        'goal_m':meta.get('goal_m'),'total_timesteps':meta.get('total_timesteps'),**hashes}
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf8')
    return archived


def creer_agent_depuis(source,nom_cible,stage_cible=None):
    """Fork a verified archived model. Never overwrite an existing target."""
    original,meta=lire_modele(source)
    target=destination_agent(nom_cible)
    out_zip,out_meta=fichiers_modele(target)
    if out_zip.exists() or out_meta.exists():
        raise FileExistsError('Destination deja presente, choisir un nouveau nom : '+str(target))
    if meta.get('control')!='vjoy_local_xyz_pedales_exclusives_v3' or meta.get('observation')!='dict_image_1x84x84_state_9' or meta.get('reset')!='csp_snapshot_verified':
        raise RuntimeError('Source incompatible avec le pilote V3.')
    current=meta.get('curriculum_stage')
    if type(current) is not int or current not in (1,2,3,4):
        raise RuntimeError('Etape source inconnue.')
    stage=current if stage_cible is None else stage_cible
    if type(stage) is not int or stage not in (current,current+1) or stage not in (1,2,3,4):
        raise ValueError('Autorise : copie de la meme etape ou transfert vers la suivante uniquement.')
    archived=archiver_agent(original,'source_transfert' if stage!=current else 'source_restauration')
    source_zip,source_meta=fichiers_modele(archived)
    # Build a new metadata tree; original metadata stays byte-for-byte intact.
    updated={**meta,'model':nom_cible,'curriculum_stage':stage,
        'goal_m':{1:10.,2:30.,3:100.,4:None}[stage],
        'parent_version':str(archived.relative_to((PROJECT/'models').resolve())),
        'parent_stage':current,'parent_total_timesteps':int(meta['total_timesteps']),
        'initialization':'stage_transfer' if stage!=current else 'restored_copy',
        'created_at':time.time(),'saved_at':time.time()}
    for key in ('run_id','log_dir'):updated.pop(key,None)
    if stage!=current:
        updated['stage_start_timesteps']=int(meta['total_timesteps'])
        updated['stage_timesteps']=0
    # Exclusive creation also prevents accidental replacement on repeated execution.
    out_zip.parent.mkdir(parents=True,exist_ok=True)
    with source_zip.open('rb') as src,out_zip.open('xb') as dst:
        shutil.copyfileobj(src,dst)
    with out_meta.open('x',encoding='utf8') as dst:
        json.dump(updated,dst,indent=2,ensure_ascii=False)
    if empreinte_fichier(out_zip)!=empreinte_fichier(source_zip):
        raise RuntimeError('Le nouvel agent ne correspond pas aux poids sources.')
    # Also freeze the initial state of the child, before its first action.
    initial=archiver_agent(target,'initialisation_agent')
    print('Agent cree :',out_zip,'| stage',stage,'| compteur herite',updated['total_timesteps'])
    print('Source conservee :',archived)
    print('Version initiale conservee :',initial)
    return target


def inventorier_versions():
    rows=[]
    for path in sorted((PROJECT/'models'/'versions').glob('*/*/manifest.json')):
        meta=json.loads(path.read_text(encoding='utf8'))
        rows.append({'modele':meta.get('model'),'stage':meta.get('curriculum_stage'),
            'compteur':meta.get('total_timesteps'),'motif':meta.get('motif'),
            'source_selectionnable':str((path.parent/'agent').relative_to(PROJECT/'models'))})
    return rows


In [ ]:
display(inventorier_versions())
if type(ACTIVER_OPERATION) is not bool:raise ValueError('ACTIVER_OPERATION doit etre True ou False.')
if not ACTIVER_OPERATION:
    print('Inventaire seulement. Activer ACTIVER_OPERATION pour appliquer le choix.')
elif OPERATION=='archiver':
    print('Version conservee :',archiver_agent(MODELE_SOURCE,'archivage_manuel'))
elif OPERATION=='revenir_version':
    creer_agent_depuis(MODELE_SOURCE,NOM_MODELE_CIBLE)
elif OPERATION=='transferer_stage':
    source,meta=lire_modele(MODELE_SOURCE)
    if STAGE_CIBLE!=meta.get('curriculum_stage',0)+1:raise ValueError('Choisir le stage suivant.')
    creer_agent_depuis(source,NOM_MODELE_CIBLE,STAGE_CIBLE)
else:
    raise ValueError('Operation inconnue.')
